In [78]:
# %% Block 0: Imports & device
import os, io, gc, json, math, random, time, zlib, shutil, csv, traceback
from collections import Counter
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torchvision
from torchvision import transforms
import torchvision.transforms.functional as TF
import timm
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cudnn.benchmark = True
print('Device:', device, f'({torch.cuda.get_device_name(0)})' if device.type == 'cuda'
      else ' <-- WARNING: no GPU.')

from kaggle_secrets import UserSecretsClient
try:
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
    print('HF_TOKEN loaded')
except Exception:
    pass

Device: cuda (Tesla T4)
HF_TOKEN loaded


In [79]:
#Check Kaggle data (runs immediately; use its output to fill Block 1 and CONFIG)
INPUT_ROOT   = '/kaggle/input'
TEST_RATIO   = 0.2
CHECK_FILES  = False    # True = fully decode every image to find broken files (slower, ~1-2 min)
IMG_EXTS     = ('.jpeg', '.jpg', '.png', '.bmp', '.webp')

def _is_img(f):
    return f.lower().endswith(IMG_EXTS)

# ---- 1. Folder tree (folders only, with file counts) ----
print(f'===== 1. Folder tree of {INPUT_ROOT} =====')
def _tree(path, depth=0, max_depth=4, max_children=6):
    try:
        entries = sorted(os.listdir(path))
    except PermissionError:
        return
    dirs = [e for e in entries if os.path.isdir(os.path.join(path, e))]
    files = [e for e in entries if not os.path.isdir(os.path.join(path, e))]
    n_img = sum(_is_img(f) for f in files)
    info = f'  ({len(files)} files, {n_img} images)' if files else ''
    print('    ' * depth + os.path.basename(path.rstrip('/')) + '/' + info)
    if depth >= max_depth:
        return
    for d in dirs[:max_children]:
        _tree(os.path.join(path, d), depth + 1, max_depth, max_children)
    if len(dirs) > max_children:
        print('    ' * (depth + 1) + f'... (+{len(dirs) - max_children} more folders)')
    for f in files:
        if not _is_img(f):
            print('    ' * (depth + 1) + f'[file] {f}')
_tree(INPUT_ROOT)

# ---- 2. Find image-folder splits: a folder whose subfolders (= classes) contain images ----
split_dirs = []
for root, dirs, files in os.walk(INPUT_ROOT):
    if root.count(os.sep) - INPUT_ROOT.count(os.sep) > 5:
        dirs[:] = []; continue
    if len(dirs) >= 2:
        sample = os.path.join(root, sorted(dirs)[0])
        if any(_is_img(f) for f in os.listdir(sample)[:50]):
            split_dirs.append(root)
            dirs[:] = []                       # don't descend into class folders
print(f'\n===== 2. Detected image-folder splits: {len(split_dirs)} =====')

# ---- 3. Per split: classes, counts, sizes, colour modes, file types ----
class_sets = {}
counts = {}
for sd in split_dirs:
    classes = sorted(d for d in os.listdir(sd) if os.path.isdir(os.path.join(sd, d)))
    per_class, modes, exts, ws, hs, broken, non_img = {}, Counter(), Counter(), [], [], [], 0
    for c in classes:
        fs = os.listdir(os.path.join(sd, c))
        imgs = [f for f in fs if _is_img(f)]
        non_img += len(fs) - len(imgs)
        per_class[c] = len(imgs)
        for f in imgs:
            p = os.path.join(sd, c, f)
            exts[os.path.splitext(f)[1].lower()] += 1
            try:
                with Image.open(p) as im:
                    ws.append(im.size[0]); hs.append(im.size[1]); modes[im.mode] += 1
                    if CHECK_FILES:
                        im.convert('RGB')
            except Exception as e:
                broken.append((p, str(e)))
    class_sets[sd], counts[sd] = classes, per_class
    total = sum(per_class.values())
    print(f'\n--- {sd}: {total} images, {len(classes)} classes ---')
    for c, n in per_class.items():
        print(f'    {c:24s} {n:6d}  {100 * n / max(total, 1):5.1f}%')
    if ws:
        print(f'    width  {min(ws)}-{max(ws)} (mean {np.mean(ws):.0f}) | height {min(hs)}-{max(hs)} (mean {np.mean(hs):.0f})')
        print(f'    shorter side: {dict(Counter(min(w, h) for w, h in zip(ws, hs)).most_common(3))} (top 3)')
    print(f'    colour modes {dict(modes)} | file types {dict(exts)} | non-image files {non_img}')
    print(f'    unreadable images: {len(broken)}' + (f' e.g. {broken[:3]}' if broken else ''))

# ---- 4. Other files (csv / txt / json) ----
print('\n===== 4. Other files =====')
for root, dirs, files in os.walk(INPUT_ROOT):
    if any(root.startswith(sd) for sd in split_dirs):
        dirs[:] = []; continue
    for f in files:
        if f.lower().endswith(('.csv', '.txt', '.json')):
            p = os.path.join(root, f)
            print(f'  {p}  ({os.path.getsize(p) / 1024:.0f} KB)')
            if f.lower().endswith('.csv'):
                with open(p) as fh:
                    for _ in range(3):
                        print('     ', fh.readline().rstrip()[:150])

# ---- 5. What was found -> copy into Block 1 by hand ----
print('\n===== 5. Copy into Block 1 =====')
for sd in split_dirs:
    print(f"  {sd}")
for parent in sorted({os.path.dirname(sd) for sd in split_dirs}):
    names = sorted(os.path.basename(sd) for sd in split_dirs if os.path.dirname(sd) == parent)
    print(f"\n  DATA_DIR = '{parent}'      # contains {names}")
for sd in split_dirs:
    print(f"  class folders in {os.path.basename(sd)}/: {class_sets[sd]}")
if len(split_dirs) >= 2:
    ref = class_sets[split_dirs[0]]
    for sd in split_dirs[1:]:
        print(f'  same classes in {os.path.basename(sd)}/ and {os.path.basename(split_dirs[0])}/: {class_sets[sd] == ref}')

===== 1. Folder tree of /kaggle/input =====
input/
    datasets/
        xbinchen/
            imagenette2-320/
                imagenette2-320/

===== 2. Detected image-folder splits: 2 =====

--- /kaggle/input/datasets/xbinchen/imagenette2-320/imagenette2-320/val: 3925 images, 10 classes ---
    n01440764                   387    9.9%
    n02102040                   395   10.1%
    n02979186                   357    9.1%
    n03000684                   386    9.8%
    n03028079                   409   10.4%
    n03394916                   394   10.0%
    n03417042                   389    9.9%
    n03425413                   419   10.7%
    n03445777                   399   10.2%
    n03888257                   390    9.9%
    width  320-1225 (mean 409) | height 320-924 (mean 351)
    shorter side: {320: 3925} (top 3)
    colour modes {'RGB': 3856, 'L': 69} | file types {'.jpeg': 3925} | non-image files 0
    unreadable images: 0

--- /kaggle/input/datasets/xbinchen/imagenette2-320/i

In [80]:
# %% Block 1: Constants (classes, models, degradation severities)
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
AVAILABLE_MODELS = ['resnet50', 'convnext_tiny', 'deit_small', 'swin_tiny', 'vgg16']

# ---- Data: written by hand (copy from the output of the data check cell) ----
DATA_DIR     = '/kaggle/input/datasets/xbinchen/imagenette2-320/imagenette2-320'
TRAIN_FOLDER = 'train'      # -> split into train + test
VAL_FOLDER   = 'val'        # -> val

# class folder -> readable name. Label order = sorted folder names.
WNID_TO_NAME = {
    'n01440764': 'tench',           'n02102040': 'english_springer',
    'n02979186': 'cassette_player', 'n03000684': 'chain_saw',
    'n03028079': 'church',          'n03394916': 'french_horn',
    'n03417042': 'garbage_truck',   'n03425413': 'gas_pump',
    'n03445777': 'golf_ball',       'n03888257': 'parachute',
}
ALL_WNIDS = sorted(WNID_TO_NAME)
NAME_TO_WNID = {v: k for k, v in WNID_TO_NAME.items()}
NAME_TO_WNID.update({'fish': 'n01440764', 'dog': 'n02102040', 'chainsaw': 'n03000684'})  # aliases

# check the hand-written values against the real folders (stops here, not hours later)
for folder in (TRAIN_FOLDER, VAL_FOLDER):
    d = os.path.join(DATA_DIR, folder)
    if not os.path.isdir(d):
        raise FileNotFoundError(f'{d} does not exist. Check DATA_DIR / folder names in Block 1.')
    missing = [w for w in ALL_WNIDS if not os.path.isdir(os.path.join(d, w))]
    if missing:
        raise FileNotFoundError(f'Class folders {missing} not found in {d}.\n'
                                f'Folders that exist there: {sorted(os.listdir(d))}')
print(f'Data OK: {DATA_DIR} | {len(ALL_WNIDS)} classes:', [WNID_TO_NAME[w] for w in ALL_WNIDS])

# Degradation table (Corruption.pdf): 5 types x 3 levels = 15 conditions + clean. Test time only.
DEGRADATIONS = ['gaussian_blur', 'gaussian_noise', 'low_light', 'jpeg', 'pixelate']
SEVERITY = {                                          # level 1 (light), 2 (medium), 3 (heavy)
    'gaussian_blur':  [1.0, 2.0, 3.0],                # sigma -> kernel 7x7, 13x13, 19x19
    'gaussian_noise': [0.04, 0.08, 0.12],             # std in [0,1] (~10, 20, 31 on 0-255)
    'low_light':      [(0.6, 1.5), (0.4, 1.9), (0.2, 2.6)],   # (f, gamma): x' = f * x^gamma
    'jpeg':           [30, 15, 7],                    # JPEG quality factor (QF)
    'pixelate':       [2, 3, 4],                      # pixel binning block size k
}
LEVEL_CODES = {                                       # names used in the report / figures
    'gaussian_blur': ['B1', 'B2', 'B3'], 'gaussian_noise': ['N1', 'N2', 'N3'],
    'low_light': ['L1', 'L2', 'L3'], 'jpeg': ['J30', 'J15', 'J7'], 'pixelate': ['D2', 'D3', 'D4'],
}
NUM_LEVELS = 3

def cond_name(d, s):
    """'clean' or the report code, e.g. ('jpeg', 2) -> 'J15'."""
    return 'clean' if d == 'clean' else LEVEL_CODES[d][s - 1]

def resolve_classes(classes):
    """'all' or a list of names/aliases/wnids -> sorted list of wnids."""
    if classes == 'all':
        return ALL_WNIDS.copy()
    out = set()
    for c in classes:
        c = c.lower()
        if c in NAME_TO_WNID: out.add(NAME_TO_WNID[c])
        elif c in WNID_TO_NAME: out.add(c)
        else: raise ValueError(f'Unknown class "{c}". Options: {list(WNID_TO_NAME.values())}')
    if len(out) < 2:
        raise ValueError('Need at least 2 classes.')
    return sorted(out)

def run_name(cfg):
    """Name of this run's result folder / zip. Auto: '<n>cls_ep<epochs>_<YYYYmmdd-HHMM>' (set once per pipeline)."""
    if not cfg.get('run_name'):
        cfg['run_name'] = (f"{len(resolve_classes(cfg['classes']))}cls_ep{cfg['epochs']}_"
                           + time.strftime('%Y%m%d-%H%M'))
    return cfg['run_name']

def get_paths(cfg):
    w = cfg['work_dir']
    p = {'work': w, 'ckpt': f'{w}/checkpoints', 'logs': f'{w}/logs', 'results': f'{w}/results/{run_name(cfg)}',
         'bench': f'{w}/benchmark', 'split': f'{w}/split.json'}
    for k in ('ckpt', 'logs', 'results', 'bench'):
        os.makedirs(p[k], exist_ok=True)
    return p

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

Data OK: /kaggle/input/datasets/xbinchen/imagenette2-320/imagenette2-320 | 10 classes: ['tench', 'english_springer', 'cassette_player', 'chain_saw', 'church', 'french_horn', 'garbage_truck', 'gas_pump', 'golf_ball', 'parachute']


In [81]:
# %% Block 2: Degradation functions (all on C x H x W tensors in [0, 1])
# Order per image: Resize 256 -> CenterCrop 224 -> [0,1] -> degradation -> clip [0,1] -> Normalize -> model

def _gaussian_blur(x, sigma):
    """2D Gaussian kernel (normalised to sum 1), size 2*ceil(3*sigma)+1, reflect padding, per channel."""
    k = 2 * math.ceil(3 * sigma) + 1
    return TF.gaussian_blur(x, kernel_size=[k, k], sigma=[sigma, sigma])

def _gaussian_noise(x, std, g):
    """x' = x + n, n ~ N(0, std^2), independent per pixel and channel."""
    return x + torch.randn(x.shape, generator=g) * std

def _low_light(x, f, gamma):
    """x' = f * x^gamma: gamma > 1 darkens shadows most, f < 1 scales the whole image down."""
    return f * x.clamp(0, 1) ** gamma

def _jpeg(x, quality):
    """Compress + decompress with libjpeg (via PIL): YCbCr, 4:2:0 chroma subsampling,
    8x8 DCT, quantisation table scaled by QF (libjpeg standard)."""
    img = TF.to_pil_image((x * 255).round().clamp(0, 255).to(torch.uint8))
    buf = io.BytesIO()
    img.save(buf, format='JPEG', quality=int(quality), subsampling=2)   # subsampling=2 -> 4:2:0
    buf.seek(0)
    return TF.to_tensor(Image.open(buf).convert('RGB'))

def _pixelate(x, k):
    """Pixel binning: average each ~k x k block (area interpolation down to round(224/k)),
    then repeat each pixel back to full size (nearest neighbour)."""
    H, W = x.shape[-2:]
    small = F.interpolate(x.unsqueeze(0), size=(round(H / k), round(W / k)), mode='area')
    return F.interpolate(small, size=(H, W), mode='nearest').squeeze(0)

def apply_degradation(x, name, level, g):
    """x: CxHxW float in [0,1]; level 1..3; g: torch.Generator (only noise is random)."""
    p = SEVERITY[name][level - 1]
    if name == 'gaussian_blur':    y = _gaussian_blur(x, p)
    elif name == 'gaussian_noise': y = _gaussian_noise(x, p, g)
    elif name == 'low_light':      y = _low_light(x, *p)
    elif name == 'jpeg':           y = _jpeg(x, p)
    elif name == 'pixelate':       y = _pixelate(x, p)
    else: raise ValueError(f'Unknown degradation: {name}. Options: {DEGRADATIONS}')
    return y.clamp(0, 1)

class FixedDegradation:
    """One (type, level). Noise seeded by image path -> identical pixels every run."""
    def __init__(self, name, severity, seed=42):
        self.name, self.severity, self.seed = name, severity, seed
    def __call__(self, x, key):
        s = zlib.crc32(f'{self.seed}|{key}|{self.name}|{self.severity}'.encode()) & 0x7FFFFFFF
        return apply_degradation(x, self.name, self.severity, torch.Generator().manual_seed(s))

In [82]:
# %% Block 3: Data (split, dataset, loaders)
SPLIT_VERSION = 3   # v3: test = val folder (unseen in pretraining), val = 20% of train folder
TRAIN_SPATIAL = transforms.Compose([transforms.RandomResizedCrop(224), transforms.RandomHorizontalFlip()])
EVAL_SPATIAL  = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224)])

def build_or_load_split(cfg):
    """<train_folder> -> train + val (stratified, val_ratio per class) ; <val_folder> -> test.
    Why: Imagenette train/ images are ImageNet-TRAIN images, which every pretrained backbone has already seen.
    Imagenette val/ images are ImageNet-VAL images (unseen) -> they must be the test set.
    Saved once as file paths in split.json. Rebuilt automatically if data / folders / classes / ratio change."""
    path = get_paths(cfg)['split']
    meta = {'version': SPLIT_VERSION, 'seed': cfg['seed'], 'val_ratio': cfg['val_ratio'],
            'data_dir': cfg['data_dir'], 'train_folder': cfg['train_folder'], 'val_folder': cfg['val_folder'],
            'classes': ALL_WNIDS}
    if os.path.exists(path):
        with open(path) as f:
            split = json.load(f)
        if all(split.get('meta', {}).get(k) == v for k, v in meta.items()):
            return split
        print('--- split.json settings changed, rebuilding ---')
    rng = random.Random(cfg['seed'])
    split = {'meta': meta, 'train': [], 'val': [], 'test': []}
    list_imgs = lambda folder, wnid: sorted(
        f'{folder}/{wnid}/{f}' for f in os.listdir(os.path.join(cfg['data_dir'], folder, wnid))
        if f.lower().endswith(('.jpeg', '.jpg', '.png')))
    for wnid in ALL_WNIDS:
        files = list_imgs(cfg['train_folder'], wnid)
        rng.shuffle(files)
        n_val = round(cfg['val_ratio'] * len(files))
        split['val'] += sorted(files[:n_val])                    # from train folder: model selection only
        split['train'] += sorted(files[n_val:])
        split['test'] += list_imgs(cfg['val_folder'], wnid)      # val folder: final evaluation
    with open(path, 'w') as f:
        json.dump(split, f, indent=1)
    print(f"Created split: train {len(split['train'])} | val {len(split['val'])} | test {len(split['test'])}")
    return split

class ImagenetteDataset(Dataset):
    """load -> spatial -> ToTensor [0,1] -> degrade (optional, LAST image op) -> normalize (optional)"""
    def __init__(self, root, rel_paths, wnids, spatial, degrade=None, normalize=True):
        label_of = {w: i for i, w in enumerate(wnids)}
        # paths look like 'train/n01440764/xxx.JPEG' -> class = 2nd part
        self.items = [(p, label_of[p.split('/')[1]]) for p in rel_paths if p.split('/')[1] in label_of]
        self.root, self.spatial, self.degrade, self.normalize = root, spatial, degrade, normalize
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        path, y = self.items[i]
        x = TF.to_tensor(self.spatial(Image.open(os.path.join(self.root, path)).convert('RGB')))
        if self.degrade is not None:
            x = self.degrade(x, path)
        if self.normalize:
            x = TF.normalize(x, MEAN, STD)
        return x, y

def make_loader(ds, cfg, shuffle=False, batch_size=None):
    nw = cfg['num_workers']
    return DataLoader(ds, batch_size=batch_size or cfg['batch_size'], shuffle=shuffle, num_workers=nw,
                      pin_memory=device.type == 'cuda', persistent_workers=shuffle and nw > 0,
                      drop_last=shuffle)

def get_dataset(cfg, split_name, spatial=EVAL_SPATIAL, degrade=None, normalize=True):
    split = build_or_load_split(cfg)
    return ImagenetteDataset(cfg['data_dir'], split[split_name], resolve_classes(cfg['classes']),
                             spatial, degrade, normalize)

In [83]:
# %% Block 4: Model factory
def get_model(name, num_classes, pretrained=True):
    if name == 'resnet50':
        m = torchvision.models.resnet50(weights='IMAGENET1K_V1' if pretrained else None, progress=False)
        m.fc = nn.Linear(m.fc.in_features, num_classes)
    elif name == 'convnext_tiny':
        m = torchvision.models.convnext_tiny(weights='IMAGENET1K_V1' if pretrained else None, progress=False)
        m.classifier[2] = nn.Linear(m.classifier[2].in_features, num_classes)
    elif name == 'vgg16':
        m = torchvision.models.vgg16(weights='IMAGENET1K_V1' if pretrained else None, progress=False)
        m.classifier[6] = nn.Linear(m.classifier[6].in_features, num_classes)
    elif name == 'deit_small':
        m = timm.create_model('deit_small_patch16_224', pretrained=pretrained, num_classes=num_classes)
    elif name == 'swin_tiny':
        m = torchvision.models.swin_t(weights='IMAGENET1K_V1' if pretrained else None, progress=False)
        m.head = nn.Linear(m.head.in_features, num_classes)
    else:
        raise ValueError(f'Unknown model {name}. Options: {AVAILABLE_MODELS}')
    return m.to(device)

In [84]:
# %% Block 5: Training (clean data, best epoch chosen on clean val)
@torch.no_grad()
def loss_and_acc(model, loader, criterion, amp):
    model.eval()
    loss_sum, correct, total = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        with torch.autocast(device.type, dtype=torch.float16, enabled=amp):
            out = model(x)
            loss = criterion(out.float(), y)
        loss_sum += loss.item() * y.size(0)
        correct += (out.argmax(1) == y).sum().item()
        total += y.size(0)
    return loss_sum / total, 100.0 * correct / total

def train_model(name, cfg):
    P = get_paths(cfg)
    wnids = resolve_classes(cfg['classes'])
    amp = cfg['amp'] and device.type == 'cuda'
    run_cfg = {k: cfg[k] for k in ('classes', 'epochs', 'batch_size', 'lr', 'weight_decay', 'seed', 'val_ratio')}
    run_cfg['split_version'] = SPLIT_VERSION
    run_cfg['classes'] = wnids
    epochs = cfg['epochs']
    verbose = cfg.get('verbose', False)
    print(f'\n[train] {name}' + (f' | {epochs} ep | bs {cfg["batch_size"]} | lr {cfg["lr"]} | {len(wnids)} classes' if verbose else ''))
    set_seed(cfg['seed'])

    train_loader = make_loader(get_dataset(cfg, 'train', spatial=TRAIN_SPATIAL), cfg, shuffle=True)
    val_loader = make_loader(get_dataset(cfg, 'val'), cfg)
    model = get_model(name, len(wnids))
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=cfg['lr'], weight_decay=cfg['weight_decay'])
    steps = len(train_loader)
    total_steps, warmup = epochs * steps, min(steps, epochs * steps // 5)   # ~1 epoch warm-up, then cosine
    lr_fn = lambda s: (s + 1) / warmup if s < warmup else \
        0.5 * (1 + math.cos(math.pi * (s - warmup) / max(1, total_steps - warmup)))
    scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_fn)
    scaler = torch.amp.GradScaler(device.type, enabled=amp)

    best_path, last_path = f"{P['ckpt']}/{name}_best.pth", f"{P['ckpt']}/{name}_last.pth"
    start, best_acc, history = 0, -1.0, []        # -1 so epoch 1 is always saved
    if os.path.exists(last_path):                     # resume / skip (Kaggle sessions can die)
        ck = torch.load(last_path, map_location=device, weights_only=False)
        if ck['config'] == run_cfg and ck['epoch'] + 1 >= epochs:
            print('--> Already trained with these settings, skipping. (Delete *_last.pth to retrain.)')
            return
        if ck['config'] == run_cfg:
            model.load_state_dict(ck['model']); optimizer.load_state_dict(ck['optimizer'])
            scheduler.load_state_dict(ck['scheduler']); scaler.load_state_dict(ck['scaler'])
            start, best_acc, history = ck['epoch'] + 1, ck['best_acc'], ck['history']
            print(f'--> Resuming from epoch {start + 1}')

    for epoch in range(start, epochs):
        model.train()
        t0, run_loss, seen = time.time(), 0.0, 0
        for i, (x, y) in enumerate(train_loader):
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device.type, dtype=torch.float16, enabled=amp):
                loss = criterion(model(x), y)
            scaler.scale(loss).backward()
            scaler.step(optimizer); scaler.update(); scheduler.step()
            run_loss += loss.item(); seen += 1
            if verbose and (i + 1) % max(1, steps // 4) == 0:
                print(f'  [ep {epoch + 1}, {i + 1:4d}/{steps}] train loss {run_loss / seen:.4f}')
                run_loss, seen = 0.0, 0
        val_loss, val_acc = loss_and_acc(model, val_loader, criterion, amp)
        history.append({'epoch': epoch + 1, 'val_loss': val_loss, 'val_acc': val_acc,
                        'lr': scheduler.get_last_lr()[0], 'time_s': round(time.time() - t0, 1)})
        is_best = val_acc > best_acc
        print(f'  ep {epoch + 1:2d}/{epochs}  val acc {val_acc:6.2f}%  {time.time() - t0:4.0f}s' + ('  *best' if is_best else ''))
        if is_best:
            best_acc = val_acc
            torch.save({'model_name': name, 'epoch': epoch + 1, 'state_dict': model.state_dict(),
                        'config': run_cfg, 'classes': wnids, 'class_names': [WNID_TO_NAME[w] for w in wnids],
                        'normalize': {'mean': MEAN, 'std': STD}, 'val_acc': val_acc}, best_path)
        torch.save({'model': model.state_dict(), 'optimizer': optimizer.state_dict(),
                    'scheduler': scheduler.state_dict(), 'scaler': scaler.state_dict(), 'epoch': epoch,
                    'best_acc': best_acc, 'history': history, 'config': run_cfg}, last_path)
        with open(f"{P['logs']}/{name}_train.json", 'w') as f:
            json.dump(history, f, indent=2)

    del model, optimizer, scaler, scheduler, train_loader, val_loader
    gc.collect()
    if device.type == 'cuda':
        torch.cuda.empty_cache()

In [85]:
# %% Block 6: Evaluation
MODEL_COLORS = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100', '#e87ba4']   # fixed order = AVAILABLE_MODELS
CM_CMAP = LinearSegmentedColormap.from_list('seq_blue', ['#fcfcfb', '#9ec5f4', '#2a78d6', '#104281'])

class Evaluator:
    """Evaluates the best checkpoints on one held-out split (default: test).

    Clean       : confusion matrix, accuracy, macro precision / recall / F1, CE loss, mean confidence
    Corruption  : for every (degradation, severity)
        Accuracy                  acc(d, s)
        Retention Rate            RR(d, s) = acc(d, s) / acc_clean
        Confidence Drop           CD(d, s) = mean max-softmax (clean) - mean max-softmax (degraded)
        Critical Severity Point   CSP(d)   = lowest severity where RR(d, s) < critical_retention
                                             (None = model stays above the threshold up to the heaviest level)
        + confusion matrix for every condition
    Each condition is built ONCE and fed to all models (the degradation is the slow, CPU part).
    """
    def __init__(self, cfg, model_names):
        self.cfg, self.P = cfg, get_paths(cfg)
        self.wnids = resolve_classes(cfg['classes'])
        self.class_names = [WNID_TO_NAME[w] for w in self.wnids]
        self.amp = cfg['amp'] and device.type == 'cuda'
        self.models = {}
        for name in model_names:
            path = f"{self.P['ckpt']}/{name}_best.pth"
            if not os.path.exists(path):
                print(f'[skip] {name}: no checkpoint at {path}'); continue
            ck = torch.load(path, map_location=device, weights_only=False)
            if ck['classes'] != self.wnids:
                print(f'[skip] {name}: trained on {ck["class_names"]}, CONFIG classes are {self.class_names}'); continue
            m = get_model(name, len(self.wnids), pretrained=False)
            m.load_state_dict(ck['state_dict']); m.eval()
            self.models[name] = m
            self.ckpt_time = getattr(self, 'ckpt_time', {}); self.ckpt_time[name] = os.path.getmtime(path)

    def conditions(self):
        return [('clean', 0)] + [(d, s) for d in self.cfg['degradations'] for s in self.cfg['severities']]

    @torch.no_grad()
    def _predict(self, loader):
        probs = {n: [] for n in self.models}
        labels = []
        for x, y in loader:
            x = x.to(device, non_blocking=True)
            for n, m in self.models.items():
                with torch.autocast(device.type, dtype=torch.float16, enabled=self.amp):
                    probs[n].append(F.softmax(m(x).float(), dim=1).cpu())
            labels.append(y)
        return {n: torch.cat(p).numpy() for n, p in probs.items()}, torch.cat(labels).numpy()

    def _metrics(self, probs, labels):
        pred = probs.argmax(1)
        p, r, f1, _ = precision_recall_fscore_support(labels, pred, labels=range(len(self.wnids)),
                                                      average='macro', zero_division=0)
        pc, rc, f1c, sup = precision_recall_fscore_support(labels, pred, labels=range(len(self.wnids)),
                                                           average=None, zero_division=0)
        return {
            'accuracy': 100.0 * float((pred == labels).mean()),
            'precision_macro': 100.0 * float(p), 'recall_macro': 100.0 * float(r), 'f1_macro': 100.0 * float(f1),
            'ce_loss': float(-np.log(probs[np.arange(len(labels)), labels] + 1e-12).mean()),
            'mean_confidence': float(probs.max(1).mean()),
            'per_class': {c: {'precision': 100.0 * float(pc[i]), 'recall': 100.0 * float(rc[i]),
                              'f1': 100.0 * float(f1c[i]), 'support': int(sup[i])}
                          for i, c in enumerate(self.class_names)},
            'confusion_matrix': confusion_matrix(labels, pred, labels=range(len(self.wnids))).tolist(),
        }

    def run(self):
        if not self.models:
            print('No models to evaluate.'); return {}
        split = self.cfg['eval_split']
        raw = {n: {} for n in self.models}
        self.probs = {n: {} for n in self.models}          # model -> condition code -> (N, C) softmax
        self.labels, self.sample_paths = None, None
        R = self.P['results']
        verbose = self.cfg.get('verbose', False)
        t_eval = time.time()
        print(f'\n[evaluate] {len(self.models)} models x {len(self.conditions())} conditions on {split}:', end=' ' if not verbose else '\n', flush=True)
        for d, s in self.conditions():
            t0 = time.time()
            code = cond_name(d, s)
            saved = [f'{R}/{n}/probs/{code}.npy' for n in self.models]
            fresh = lambda n, p: os.path.exists(p) and os.path.getmtime(p) >= self.ckpt_time[n]   # newer than the model
            if os.path.exists(f'{R}/labels.npy') and all(fresh(n, p) for n, p in zip(self.models, saved)):
                # already on disk from an earlier (interrupted) run with the same run_name -> reuse
                labels = np.load(f'{R}/labels.npy')
                probs = {n: np.load(p) for n, p in zip(self.models, saved)}
                if self.sample_paths is None:
                    with open(f'{R}/sample_paths.json') as f:
                        self.sample_paths = json.load(f)
                note = 'loaded from disk'
            else:
                deg = None if d == 'clean' else FixedDegradation(d, s, self.cfg['seed'])
                ds = get_dataset(self.cfg, split, degrade=deg)
                probs, labels = self._predict(make_loader(ds, self.cfg))
                np.save(f'{R}/labels.npy', labels.astype(np.int64))          # always the current test set
                with open(f'{R}/sample_paths.json', 'w') as f:
                    json.dump([p for p, _ in ds.items], f)
                for n, p in zip(self.models, saved):            # save NOW: survives a later crash
                    os.makedirs(os.path.dirname(p), exist_ok=True)
                    np.save(p, probs[n].astype(np.float32))
                note = f'{time.time() - t0:.0f}s'
            if self.labels is None:
                self.labels = labels
                if self.sample_paths is None:
                    with open(f'{R}/sample_paths.json') as f:
                        self.sample_paths = json.load(f)
            key = 'clean' if d == 'clean' else f'{d}_s{s}'
            for n in self.models:
                raw[n][key] = self._metrics(probs[n], labels)
                self.probs[n][code] = probs[n]
            if verbose:
                print(f'  {cond_name(d, s):6s} {key:18s} ' + ' | '.join(f'{n} {raw[n][key]["accuracy"]:.1f}%' for n in self.models)
                      + f'  ({note})')
            else:
                print(code, end=' ', flush=True)
        results = {n: self._derive(raw[n]) for n in self.models}
        self._save(results)
        if not verbose:
            print(f'({(time.time() - t_eval) / 60:.1f} min)')
        self.print_summary(results, verbose)
        return results

    def _derive(self, raw):
        clean = raw['clean']
        th = self.cfg['critical_retention']
        corr = {}
        for d in self.cfg['degradations']:
            rows = []
            for s in self.cfg['severities']:
                m = raw[f'{d}_s{s}']
                rows.append({'severity': s, 'code': cond_name(d, s), 'accuracy': m['accuracy'],
                             'retention_rate': m['accuracy'] / clean['accuracy'] if clean['accuracy'] > 0 else 0.0,
                             'confidence': m['mean_confidence'],
                             'confidence_drop': clean['mean_confidence'] - m['mean_confidence'],
                             'ce_loss': m['ce_loss'], 'f1_macro': m['f1_macro'],
                             'confusion_matrix': m['confusion_matrix']})
            below = [r['severity'] for r in rows if r['retention_rate'] < th]
            corr[d] = {'by_severity': rows,
                       'mean_accuracy': float(np.mean([r['accuracy'] for r in rows])),
                       'mean_retention_rate': float(np.mean([r['retention_rate'] for r in rows])),
                       'mean_confidence_drop': float(np.mean([r['confidence_drop'] for r in rows])),
                       'critical_severity_point': min(below) if below else None}
        return {'clean': clean, 'corruption': corr,
                'overall': {'mean_corrupted_accuracy': float(np.mean([v['mean_accuracy'] for v in corr.values()])),
                            'mean_retention_rate': float(np.mean([v['mean_retention_rate'] for v in corr.values()])),
                            'mean_confidence_drop': float(np.mean([v['mean_confidence_drop'] for v in corr.values()]))},
                'settings': {'split': self.cfg['eval_split'], 'critical_retention': th,
                             'classes': self.class_names}}

    # ---------- saving ----------
    def _save(self, results):
        """results/<run_name>/
             run_info.json            config, classes, condition order, environment
             labels.npy               (N,)   true label index per test image
             sample_paths.json        image path per row of labels.npy / probs
             summary.csv              one row per model
             by_severity.csv          one row per model x condition (long format)
             <model>/metrics.json     all metrics (confusion matrices live in the .npy files)
             <model>/confusion/<code>.npy     (C, C) int counts, rows = true, cols = predicted
             <model>/confusion_all.npy        (K, C, C) stacked in run_info['conditions'] order
             <model>/probs/<code>.npy         (N, C) float32 softmax per test image (written per condition)
             <model>/train_history.json       per-epoch val loss / acc (if trained)
             figures/                          quick-look PNGs (if CONFIG['save_plots'])"""
        R = self.P['results']
        codes = [cond_name(d, s) for d, s in self.conditions()]
        conds = [{'code': cond_name(d, s), 'degradation': d, 'severity': s,
                  'param': None if d == 'clean' else SEVERITY[d][s - 1]} for d, s in self.conditions()]
        with open(f'{R}/run_info.json', 'w') as f:
            json.dump({'run_name': run_name(self.cfg),
                       'created': time.strftime('%Y-%m-%d %H:%M:%S'),
                       'eval_split': self.cfg['eval_split'], 'n_samples': int(len(self.labels)),
                       'classes_wnid': self.wnids, 'class_names': self.class_names,
                       'models': list(results), 'conditions': conds,
                       'critical_retention': self.cfg['critical_retention'],
                       'config': {k: v for k, v in self.cfg.items() if k != 'degradations'} | {'degradations': list(self.cfg['degradations'])},
                       'environment': {'torch': torch.__version__, 'torchvision': torchvision.__version__,
                                       'timm': timm.__version__, 'device': str(device),
                                       'gpu': torch.cuda.get_device_name(0) if device.type == 'cuda' else None}},
                      f, indent=1, default=str)

        strip = lambda o: ({k: strip(v) for k, v in o.items() if k != 'confusion_matrix'} if isinstance(o, dict)
                           else [strip(v) for v in o] if isinstance(o, list) else o)
        for n, res in results.items():
            M = f'{R}/{n}'
            os.makedirs(f'{M}/confusion', exist_ok=True); os.makedirs(f'{M}/probs', exist_ok=True)
            cms = [np.array(res['clean']['confusion_matrix'], dtype=np.int64)]
            np.save(f'{M}/confusion/clean.npy', cms[0])
            for d, v in res['corruption'].items():
                for row in v['by_severity']:
                    cm = np.array(row['confusion_matrix'], dtype=np.int64)
                    np.save(f"{M}/confusion/{row['code']}.npy", cm); cms.append(cm)
            np.save(f'{M}/confusion_all.npy', np.stack(cms))
            for code in codes:
                np.save(f'{M}/probs/{code}.npy', self.probs[n][code])
            with open(f'{M}/metrics.json', 'w') as f:
                json.dump(strip(res), f, indent=1)
            hist = f"{self.P['logs']}/{n}_train.json"
            if os.path.exists(hist):
                shutil.copy2(hist, f'{M}/train_history.json')

        # one row per model (clean + overall + CSP per degradation)
        with open(f'{R}/summary.csv', 'w', newline='') as f:
            w = csv.writer(f)
            w.writerow(['model', 'clean_acc', 'precision', 'recall', 'f1', 'clean_ce_loss', 'mean_corrupted_acc',
                        'mean_retention', 'mean_conf_drop'] + [f'CSP_{d}' for d in self.cfg['degradations']])
            for n, r in results.items():
                c, o = r['clean'], r['overall']
                w.writerow([n] + [round(v, 4) for v in (c['accuracy'], c['precision_macro'], c['recall_macro'],
                            c['f1_macro'], c['ce_loss'], o['mean_corrupted_accuracy'], o['mean_retention_rate'],
                            o['mean_confidence_drop'])] +
                           [r['corruption'][d]['critical_severity_point'] or '-' for d in self.cfg['degradations']])
        # long format: one row per model x condition (easy to pivot / plot later)
        with open(f'{R}/by_severity.csv', 'w', newline='') as f:
            w = csv.writer(f)
            w.writerow(['model', 'degradation', 'severity', 'code', 'accuracy', 'retention_rate', 'confidence',
                        'confidence_drop', 'ce_loss', 'f1_macro'])
            for n, r in results.items():
                w.writerow([n, 'clean', 0, 'clean', r['clean']['accuracy'], 1.0, r['clean']['mean_confidence'], 0.0,
                            r['clean']['ce_loss'], r['clean']['f1_macro']])
                for d, v in r['corruption'].items():
                    for row in v['by_severity']:
                        w.writerow([n, d, row['severity'], row['code'], row['accuracy'], row['retention_rate'], row['confidence'],
                                    row['confidence_drop'], row['ce_loss'], row['f1_macro']])
        if self.cfg.get('save_plots', True):
            os.makedirs(f'{R}/figures', exist_ok=True)
            for n, res in results.items():
                self.plot_confusions(n, res)
            self.plot_curves(results)
        if self.cfg.get('verbose', False):
            print(f'Saved results to {R}')

    # ---------- plots ----------
    def _draw_cm(self, ax, cm, title):
        cm = np.array(cm)
        norm = cm / np.maximum(cm.sum(1, keepdims=True), 1)
        ax.imshow(norm, cmap=CM_CMAP, vmin=0, vmax=1)
        k = len(self.class_names)
        ax.set_xticks(range(k)); ax.set_yticks(range(k))
        ax.set_xticklabels(self.class_names, rotation=45, ha='right', fontsize=7)
        ax.set_yticklabels(self.class_names, fontsize=7)
        for i in range(k):
            for j in range(k):
                if cm[i, j]:
                    ax.text(j, i, cm[i, j], ha='center', va='center', fontsize=6,
                            color='#ffffff' if norm[i, j] > 0.55 else '#0b0b0b')
        ax.set_title(title, fontsize=9); ax.set_xlabel('predicted', fontsize=8); ax.set_ylabel('true', fontsize=8)

    def plot_confusions(self, name, res):
        """<model>_cm.png: clean matrix + one row per degradation (all severities). Rows normalised, counts shown."""
        degs, sevs = self.cfg['degradations'], self.cfg['severities']
        cols = max(len(sevs), 1)
        fig, axes = plt.subplots(len(degs) + 1, cols, figsize=(3.2 * cols, 3.2 * (len(degs) + 1)), squeeze=False)
        for ax in axes.flat:
            ax.axis('off')
        axes[0, 0].axis('on')
        self._draw_cm(axes[0, 0], res['clean']['confusion_matrix'], f'clean | acc {res["clean"]["accuracy"]:.1f}%')
        for i, d in enumerate(degs):
            for j, row in enumerate(res['corruption'][d]['by_severity']):
                axes[i + 1, j].axis('on')
                self._draw_cm(axes[i + 1, j], row['confusion_matrix'],
                              f'{row["code"]} ({d}) | acc {row["accuracy"]:.1f}%')
        fig.suptitle(f'{name}: confusion matrices ({self.cfg["eval_split"]} split)', fontsize=12)
        fig.tight_layout()
        fig.savefig(f"{self.P['results']}/figures/{name}_cm.png", dpi=110)
        plt.close(fig)

    def plot_curves(self, results):
        """curves.png: accuracy and retention rate vs severity, one panel per degradation, one line per model."""
        degs = self.cfg['degradations']
        fig, axes = plt.subplots(2, len(degs), figsize=(3.4 * len(degs), 6.4), squeeze=False, sharey='row')
        for n, r in results.items():
            color = MODEL_COLORS[AVAILABLE_MODELS.index(n) % len(MODEL_COLORS)]
            for j, d in enumerate(degs):
                rows = r['corruption'][d]['by_severity']
                xs = [0] + [row['severity'] for row in rows]
                axes[0, j].plot(xs, [r['clean']['accuracy']] + [row['accuracy'] for row in rows],
                                color=color, lw=2, marker='o', ms=5, label=n)
                axes[1, j].plot(xs, [1.0] + [row['retention_rate'] for row in rows],
                                color=color, lw=2, marker='o', ms=5, label=n)
        for j, d in enumerate(degs):
            axes[0, j].set_title(d, fontsize=10)
            axes[1, j].axhline(self.cfg['critical_retention'], color='#898781', lw=1, ls='--')
            for i in range(2):
                ax = axes[i, j]
                sev = self.cfg['severities']
                ax.set_xticks([0] + list(sev))
                ax.set_xticklabels(['clean'] + [cond_name(d, s) for s in sev], fontsize=8)
                ax.grid(alpha=0.25, lw=0.6)
                for side in ('top', 'right'):
                    ax.spines[side].set_visible(False)
            axes[1, j].set_xlabel('level')
        axes[0, 0].set_ylabel('accuracy (%)')
        axes[1, 0].set_ylabel('retention rate')
        fig.text(0.5, 0.005, f'dashed line = CSP threshold (retention {self.cfg["critical_retention"]})',
                 ha='center', fontsize=8, color='#52514e')
        axes[0, -1].legend(fontsize=8, frameon=False)
        fig.tight_layout()
        fig.savefig(f"{self.P['results']}/figures/curves.png", dpi=120)
        plt.close(fig)

    @staticmethod
    def print_summary(results, verbose=False):
        if not results:
            return
        clean = {n: r['clean']['accuracy'] for n, r in results.items()}
        corr = {n: r['overall']['mean_corrupted_accuracy'] for n, r in results.items()}
        bc, bk = max(clean, key=clean.get), max(corr, key=corr.get)
        print('\n=========== RESULT ===========')
        print(f'Best on clean     : {bc:14s} acc {clean[bc]:.2f}%')
        n_cond = sum(len(v['by_severity']) for v in results[bk]['corruption'].values())
        print(f'Best on corrupted : {bk:14s} mean acc {corr[bk]:.2f}% over {n_cond} conditions '
              f"(retention {results[bk]['overall']['mean_retention_rate']:.3f})")
        if not verbose:
            return
        degs = list(next(iter(results.values()))['corruption'])
        print('\n' + '=' * 30 + ' CLEAN ' + '=' * 30)
        print(f"{'model':14s} {'acc':>7s} {'prec':>7s} {'recall':>7s} {'F1':>7s} {'CE':>7s}")
        for n, r in results.items():
            c = r['clean']
            print(f"{n:14s} {c['accuracy']:7.2f} {c['precision_macro']:7.2f} {c['recall_macro']:7.2f} "
                  f"{c['f1_macro']:7.2f} {c['ce_loss']:7.4f}")
        print('\n' + '=' * 26 + ' CORRUPTION ' + '=' * 26)
        print(f"{'model':14s} {'mCorrAcc':>9s} {'mRetent':>8s} {'mConfDrop':>10s}   CSP per degradation")
        for n, r in results.items():
            o = r['overall']
            csp = ', '.join(f"{d}={r['corruption'][d]['critical_severity_point'] or '-'}" for d in degs)
            print(f"{n:14s} {o['mean_corrupted_accuracy']:9.2f} {o['mean_retention_rate']:8.3f} "
                  f"{o['mean_confidence_drop']:10.4f}   {csp}")

In [86]:
# %% Block 7: Data preparation (degradation preview + export of degraded sets)
def preview_degradations(cfg):
    """degradation_preview.png: rows = degradation types, cols = clean + level 1..3."""
    ds = get_dataset(cfg, cfg['eval_split'], normalize=False)
    x, _ = ds[0]
    key = ds.items[0][0]
    rows = [torch.stack([x] + [FixedDegradation(d, s, cfg['seed'])(x, key) for s in range(1, NUM_LEVELS + 1)])
            for d in DEGRADATIONS]
    out = f"{cfg['work_dir']}/degradation_preview.png"
    torchvision.utils.save_image(torch.cat(rows), out, nrow=NUM_LEVELS + 1, padding=4, pad_value=1.0)
    if cfg.get('verbose', False):
        print(f'Saved {out} (rows: {", ".join(DEGRADATIONS)}; cols: clean, levels 1..3)')

def export_degraded(cfg, split_name):
    """benchmark/<split>/: clean + every (degradation, severity) as uint8 tensors (N,3,224,224) + manifest.
    To use: x = images.float() / 255 -> Normalize(MEAN, STD) -> model."""
    out_dir = f"{get_paths(cfg)['bench']}/{split_name}"
    os.makedirs(out_dir, exist_ok=True)
    conds = [('clean', 0)] + [(d, s) for d in cfg['degradations'] for s in cfg['severities']]
    files = {}
    print(f'\n=== EXPORT {split_name} split ({len(conds)} conditions) ===')
    for d, s in conds:
        deg = None if d == 'clean' else FixedDegradation(d, s, cfg['seed'])
        ds = get_dataset(cfg, split_name, degrade=deg, normalize=False)
        xs, ys = [], []
        for x, y in make_loader(ds, cfg, batch_size=64):
            xs.append((x * 255).round().to(torch.uint8)); ys.append(y)
        fname = 'clean.pt' if d == 'clean' else f'{d}_{cond_name(d, s)}.pt'
        torch.save({'images': torch.cat(xs), 'labels': torch.cat(ys), 'paths': [p for p, _ in ds.items],
                    'degradation': d, 'severity': s, 'code': cond_name(d, s)}, f'{out_dir}/{fname}')
        files[fname] = {'degradation': d, 'severity': s, 'n': len(ds)}
        print(f'  {fname:28s} {len(ds)} images')
    wnids = resolve_classes(cfg['classes'])
    with open(f'{out_dir}/manifest.json', 'w') as f:
        json.dump({'split': split_name, 'classes_wnid': wnids, 'class_names': [WNID_TO_NAME[w] for w in wnids],
                   'preprocess': 'Resize(256) -> CenterCrop(224) -> degradation in [0,1] -> uint8',
                   'model_input': 'float/255 -> Normalize(mean, std)', 'normalize': {'mean': MEAN, 'std': STD},
                   'severity_params': SEVERITY, 'seed': cfg['seed'], 'files': files}, f, indent=2)

In [87]:
# %% Block 8: Pipeline runner
def restore_previous_outputs(cfg):
    """Copy checkpoints / logs / split from a previous Kaggle version (added as input). Skips benchmark/."""
    src = cfg.get('resume_dir')
    if not src or not os.path.isdir(src):
        return
    n = 0
    for root, _, files in os.walk(src):
        rel = os.path.relpath(root, src)
        if rel.split(os.sep)[0] == 'benchmark':
            continue
        for f in files:
            dst = os.path.join(cfg['work_dir'], rel, f)
            if not os.path.exists(dst):
                os.makedirs(os.path.dirname(dst), exist_ok=True)
                shutil.copy2(os.path.join(root, f), dst); n += 1
    print(f'--- Restored {n} files from {src} ---')

def package_results(cfg):
    """Zip results/<run_name> (+ split.json, preview) into /kaggle/working/<run_name>.zip.
    Checkpoints are left out (large); set CONFIG['zip_checkpoints'] = True to include the *_best.pth files."""
    import zipfile
    P = get_paths(cfg)
    name = run_name(cfg)
    out = f"{cfg['work_dir']}/{name}.zip"
    with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
        for root, _, files in os.walk(P['results']):
            for f in files:
                full = os.path.join(root, f)
                z.write(full, os.path.join(name, os.path.relpath(full, P['results'])))
        for f in sorted(os.listdir(P['logs'])):                      # per-epoch training history
            z.write(os.path.join(P['logs'], f), os.path.join(name, 'logs', f))
        for extra in (P['split'], f"{cfg['work_dir']}/degradation_preview.png"):
            if os.path.exists(extra):
                z.write(extra, os.path.join(name, os.path.basename(extra)))
        if cfg.get('zip_checkpoints'):
            for m in cfg['models']:
                ck = f"{P['ckpt']}/{m}_best.pth"
                if os.path.exists(ck):
                    z.write(ck, os.path.join(name, 'checkpoints', os.path.basename(ck)))
    print(f'[saved] {out} ({os.path.getsize(out) / 1e6:.1f} MB)')

def run_pipeline(cfg):
    t_start = time.time()
    cfg = dict(cfg)
    print(f"Run: {run_name(cfg)}")
    cfg['models'] = AVAILABLE_MODELS.copy() if cfg['models'] == 'all' else list(cfg['models'])
    print(f"{len(resolve_classes(cfg['classes']))} classes | models {cfg['models']} | {cfg['epochs']} epochs | steps {cfg['steps']}")
    get_paths(cfg)
    restore_previous_outputs(cfg)
    build_or_load_split(cfg)
    results = None
    for step in cfg['steps']:
        try:
            if step == 'preview':
                preview_degradations(cfg)
            elif step == 'train':
                for m in cfg['models']:
                    try:
                        train_model(m, cfg)
                    except Exception:            # one failing model must not kill an unattended run
                        traceback.print_exc(); print(f'[ERROR] training {m} failed, continuing')
                package_results(cfg)             # early zip: training logs are safe even if evaluation fails
            elif step == 'evaluate':
                results = Evaluator(cfg, cfg['models']).run()
            elif step == 'export':
                for sp in cfg['export_splits']:
                    export_degraded(cfg, sp)
            elif step == 'package':
                pass                             # packaging always runs at the end (see below)
            else:
                print(f'[WARN] unknown step "{step}"')
        except Exception:
            traceback.print_exc(); print(f'[ERROR] step "{step}" failed, continuing')
    try:
        package_results(cfg)                     # final zip, whatever happened above
    except Exception:
        traceback.print_exc()
    print(f'\nDone in {(time.time() - t_start) / 60:.1f} min.')
    return results

In [91]:
# %% Block 9: CONFIG + RUN  (the only block you edit)
CONFIG = {
    # ---- paths ----
    'data_dir':     DATA_DIR,       # set by hand in Block 1
    'train_folder': TRAIN_FOLDER,   # set by hand in Block 1
    'val_folder':   VAL_FOLDER,     # set by hand in Block 1
    'work_dir':   '/kaggle/working',
    'resume_dir': None,          # previous version's output (Add Input -> Your Work), e.g. '/kaggle/input/<notebook>'
    'run_name':   'run_fix_data_leak',   # results folder + zip name (keep it fixed so a rerun can reuse saved results)

    # ---- what to run (in this order) ----
    # 'preview'  degradation example grid      'train'   fine-tune models
    # 'evaluate' clean + corruption metrics     'export'  save degraded copies of export_splits (.pt)
    # results are zipped automatically -> /kaggle/working/<run_name>.zip (after training and at the end)
    # corrupted images are never written to disk: they are re-created from the algorithms (Block 2)
    'steps': ['preview', 'train', 'evaluate'],

    # ---- data ----
    'classes':    'all',                     # or e.g. ['fish', 'french_horn']
    'val_ratio': 0.2,                       # share of train/ per class held out as test
    'seed':       42,

    # ---- models & training ----
    'models':       ['resnet50', 'convnext_tiny', 'deit_small', 'swin_tiny'],     # or e.g. ['resnet50', 'swin_tiny']
    'epochs':       10,
    'batch_size':   64,
    'lr':           1e-4,
    'weight_decay': 0.05,
    'num_workers':  4,
    'amp':          True,      # mixed precision on GPU

    # ---- evaluation / degradation ----
    'eval_split':         'test',
    'degradations':       DEGRADATIONS,       # subset allowed, e.g. ['gaussian_blur', 'jpeg']
    'severities':         [1, 2, 3],          # 1 = light, 2 = medium, 3 = heavy
    'critical_retention': 0.90,              # CSP = first severity where retention rate < this
    'export_splits':      ['val'],           # used by the 'export' step

    # ---- outputs ----
    'verbose':         False,   # True = batch losses, per-condition accuracies, full tables in the log
    'save_plots':      True,    # quick-look PNGs on Kaggle (all numbers are saved as .npy/.csv/.json anyway)
    'zip_checkpoints': False,   # True = also put *_best.pth in the zip (~100-500 MB per model)
}

results = run_pipeline(CONFIG)

Run: run_fix_data_leak
10 classes | models ['resnet50', 'convnext_tiny', 'deit_small', 'swin_tiny'] | 10 epochs | steps ['preview', 'train', 'evaluate']

[train] resnet50
--> Already trained with these settings, skipping. (Delete *_last.pth to retrain.)

[train] convnext_tiny
--> Already trained with these settings, skipping. (Delete *_last.pth to retrain.)

[train] deit_small
--> Already trained with these settings, skipping. (Delete *_last.pth to retrain.)

[train] swin_tiny
--> Already trained with these settings, skipping. (Delete *_last.pth to retrain.)
[saved] /kaggle/working/run_fix_data_leak.zip (12.3 MB)

[evaluate] 4 models x 16 conditions on test: clean B1 B2 B3 N1 N2 N3 L1 L2 L3 J30 J15 J7 D2 D3 D4 (0.2 min)

=========== RESULT ===========
Best on clean     : convnext_tiny  acc 99.69%
Best on corrupted : deit_small     mean acc 94.83% over 15 conditions (retention 0.956)
[saved] /kaggle/working/run_fix_data_leak.zip (12.3 MB)

Done in 0.3 min.
